# Week 4 Coding Assignment: Data Acquisition — SQL, APIs & Web Data

**Data Science: Foundations to Applications (SCD-4XX) | Kyungdong University, Global Campus**

**Outcome being assessed:** Retrieve data from databases and external sources programmatically.

## Instructions
- Complete every function below by replacing `# TODO` with your code.
- **Always use `?` placeholders in SQL queries that include a variable — never an f-string or `%` formatting.** This is graded as part of the assignment.
- Run the test cell below each function to check your answer.
- When finished: save the notebook, then `git add`, `git commit`, and `git push` to your `data-science-labs-2026` repository, per the Week 1 lab.

**Parts:**
- Part A — SQL Fundamentals (Problems 1–4)
- Part B — Python + SQL Safely (Problems 5–6)
- Part C — REST APIs (Problems 7–9)
- Stretch Challenge (optional) — Ethical Web Scraping


> ⚠️ **Run this notebook in Jupyter Notebook (via Anaconda) — not Google Colab.**
This course uses your local Jupyter environment for all labs and assignments.
Open it the same way you did in Week 1: navigate to your cloned repo folder in a terminal and run `jupyter notebook`.

## Setup — run this first
This builds a sample `retail.db` SQLite database with two linked tables, `customers` and `orders`, that every problem below uses. You don't need to modify this cell.

In [37]:
import sqlite3
import random
import time
import requests
import pandas as pd
from datetime import date, timedelta

random.seed(7)

conn = sqlite3.connect("retail.db")
cur = conn.cursor()

cur.execute("DROP TABLE IF EXISTS orders")
cur.execute("DROP TABLE IF EXISTS customers")

cur.execute("""
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    name TEXT,
    city TEXT,
    segment TEXT
)
""")
cur.execute("""
CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    order_date TEXT,
    amount REAL,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
)
""")

cities = ["Seoul", "Busan", "Incheon", "Daegu", "Gwangju"]
segments = ["Retail", "Wholesale", "Online"]
names = ["Kim Minji","Park Jisoo","Lee Jaehyun","Choi Seoyeon","Jung Hyunwoo",
         "Kang Yuna","Yoon Doyoon","Han Soomin","Oh Jiwoo","Shin Taeyang",
         "Baek Narae","Song Junho","Moon Hana","Ahn Jiho","Jang Yerin"]

customers = [(i, name, random.choice(cities), random.choice(segments))
             for i, name in enumerate(names, start=1)]
cur.executemany("INSERT INTO customers VALUES (?,?,?,?)", customers)

start_date = date(2026, 1, 1)
order_id, orders = 1, []
for c in customers:
    for _ in range(random.randint(2, 8)):
        d = start_date + timedelta(days=random.randint(0, 270))
        amt = round(random.uniform(15000, 450000), -2)
        orders.append((order_id, c[0], d.isoformat(), amt))
        order_id += 1
cur.executemany("INSERT INTO orders VALUES (?,?,?,?)", orders)
conn.commit()

print(f"retail.db ready: {len(customers)} customers, {len(orders)} orders.")

retail.db ready: 15 customers, 73 orders.


---
## Part A — SQL Fundamentals

### Problem 1 — Find Customers by City and Segment
Return the `name`, `city`, and `segment` of every customer in a given `city` and `segment`.

**Hint:** `WHERE ... AND ...`

In [38]:
def customers_in_city_segment(conn, city, segment):
    """
    conn: sqlite3 connection
    city: str, e.g. "Seoul"
    segment: str, e.g. "Online"
    returns: pandas DataFrame with columns name, city, segment
    """
    # TODO: implement using pd.read_sql and a parameterized query (use ? placeholders!)
    pass

# Test

def customers_in_city_segment(conn, city, segment):
    sql = """
        SELECT name, city, segment
        FROM customers
        WHERE city = ? AND segment = ?
    """
    return pd.read_sql_query(sql, conn, params=(city, segment))


### Problem 2 — Join Customers with Their Orders
Return `name`, `order_date`, and `amount` for every order, joined with the customer who placed it. Limit to the first 10 rows.

**Hint:** `INNER JOIN ... ON ...`, `LIMIT`

In [39]:
def orders_with_customer_names(conn):
    """
    returns: pandas DataFrame with columns name, order_date, amount (first 10 rows)
    """
    # TODO: implement
    pass


def orders_with_customer_names(conn):
    sql = """
        SELECT c.name, o.order_date, o.amount
        FROM orders AS o
        INNER JOIN customers AS c
            ON o.customer_id = c.customer_id
        ORDER BY o.order_id
        LIMIT 10
    """
    return pd.read_sql_query(sql, conn)

### Problem 3 — Total Revenue per Customer (High Spenders Only)
Return each customer's `name` and their `total_spent` (sum of `amount`), but only for customers whose total exceeds a given `min_total`. Sort by `total_spent` descending.

**Hint:** `GROUP BY`, `HAVING`, `ORDER BY ... DESC`

In [40]:
def high_spending_customers(conn, min_total=300000):
    """
    returns: pandas DataFrame with columns name, total_spent
    """
    # TODO: implement
    pass


def high_spending_customers(conn, min_total=300000):
    sql = """
        SELECT c.name, SUM(o.amount) AS total_spent
        FROM customers AS c
        INNER JOIN orders AS o
            ON c.customer_id = o.customer_id
        GROUP BY c.customer_id, c.name
        HAVING SUM(o.amount) > ?
        ORDER BY total_spent DESC
    """
    return pd.read_sql_query(
        sql, conn, params=(min_total,)
    )

### Problem 4 — Top N Customers by Spend
Return the top `n` customers by total spend — same idea as Problem 3, but capped to a specific count regardless of threshold.

**Hint:** `LIMIT`

In [41]:
def top_n_customers(conn, n=3):
    """
    returns: pandas DataFrame with columns name, total_spent — exactly n rows
    """
    # TODO: implement
    pass


def top_n_customers(conn, n=3):
    sql = """
        SELECT c.name, SUM(o.amount) AS total_spent
        FROM customers AS c
        INNER JOIN orders AS o
            ON c.customer_id = o.customer_id
        GROUP BY c.customer_id, c.name
        ORDER BY total_spent DESC
        LIMIT ?
    """
    return pd.read_sql_query(sql, conn, params=(n,))

---
## Part B — Python + SQL Safely

### Problem 5 — Search Orders by Customer Name (Safely!)
Write a function that looks up every order placed by a given customer name. **This must use a `?` placeholder** — it will be checking user-supplied input, exactly the SQL-injection scenario from the demo.

**Hint:** `cursor.execute(sql, (value,))`, then `.fetchall()`

In [42]:
def find_orders_by_customer(conn, customer_name):
    """
    conn: sqlite3 connection
    customer_name: str
    returns: list of tuples (order_id, order_date, amount)
    """
    # TODO: implement using cursor.execute with a ? placeholder — no f-strings!
    pass


def find_orders_by_customer(conn, customer_name):
    sql = """
        SELECT o.order_id, o.order_date, o.amount
        FROM orders AS o
        INNER JOIN customers AS c
            ON o.customer_id = c.customer_id
        WHERE c.name = ?
        ORDER BY o.order_id
    """
    cursor = conn.cursor()
    cursor.execute(sql, (customer_name,))
    return cursor.fetchall()

### Problem 6 — Monthly Revenue Report
Using `pd.read_sql()`, return total revenue grouped by month (format `YYYY-MM`), ordered chronologically.

**Hint:** SQLite's `strftime('%Y-%m', order_date)` extracts the year-month from a date string.

In [43]:
def monthly_revenue(conn):
    """
    returns: pandas DataFrame with columns month, revenue
    """
    # TODO: implement using pd.read_sql
    pass


def monthly_revenue(conn):
    sql = """
        SELECT
            strftime('%Y-%m', order_date) AS month,
            SUM(amount) AS revenue
        FROM orders
        GROUP BY strftime('%Y-%m', order_date)
        ORDER BY month
    """
    return pd.read_sql_query(sql, conn)

---
## Part C — REST APIs

### Problem 7 — Fetch Weather for a City
Write a function that calls the Open-Meteo API (same one from the demo) for a given latitude/longitude and returns the parsed JSON `current_weather` dictionary. Check `response.status_code` and return `None` if the request fails.

**Hint:** `requests.get(url, params=...)`, `response.json()`

In [44]:
def get_current_weather(latitude, longitude):
    """
    returns: dict (the current_weather portion of the response), or None on failure
    """
    url = "https://api.open-meteo.com/v1/forecast"
    # TODO: build params, send the GET request, check status_code, return data["current_weather"]
    pass


def get_current_weather(latitude, longitude):
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "current_weather": "true"
    }

    try:
        response = requests.get(
            url, params=params, timeout=15
        )
        response.raise_for_status()
        data = response.json()
        return data.get("current_weather")
    except (requests.RequestException, ValueError):
        return None

### Problem 8 — Weather for Multiple Cities into a DataFrame
Using the function from Problem 7, fetch the current weather for a list of `(city_name, lat, lon)` tuples and combine the results into a single DataFrame with a `city` column.

**Hint:** Build a list of dicts (adding a `"city"` key to each result), then `pd.DataFrame(list_of_dicts)`.

In [45]:
def weather_for_cities(cities):
    """
    cities: list of (city_name, latitude, longitude) tuples
    returns: pandas DataFrame with a 'city' column plus the weather fields
    """
    # TODO: implement — loop over cities, call get_current_weather, collect into a DataFrame
    pass

test_cities = [("Seoul", 37.5665, 126.9780), ("Busan", 35.1796, 129.0756), ("Tokyo", 35.6762, 139.6503)]

def weather_for_cities(cities):
    rows = []

    for city_name, latitude, longitude in cities:
        weather = get_current_weather(latitude, longitude)

        row = {"city": city_name}

        if weather is not None:
            row.update(weather)

        rows.append(row)

    return pd.DataFrame(rows)

### Problem 9 — Respect the Rate Limit
Modify your approach from Problem 8 so that your function pauses `delay_seconds` between each API call, exactly as discussed in the demo's "Authentication and Rate Limits" section.

**Hint:** `time.sleep(delay_seconds)` inside your loop, after each request.

In [46]:
def weather_for_cities_paced(cities, delay_seconds=1):
    """
    Same as Problem 8, but pauses delay_seconds between each request.
    returns: pandas DataFrame with a 'city' column plus the weather fields
    """
    # TODO: implement
    pass


import time

def weather_for_cities_paced(cities, delay_seconds=1):
    rows = []

    for city_name, latitude, longitude in cities:
        weather = get_current_weather(latitude, longitude)

        row = {"city": city_name}

        if weather is not None:
            row.update(weather)

        rows.append(row)
        time.sleep(delay_seconds)

    return pd.DataFrame(rows)

---
## Stretch Challenge (Optional) — Ethical Web Scraping

Scrape the first page of [books.toscrape.com](http://books.toscrape.com) and return a DataFrame with each book's title and price. Follow the demo's ethics checklist: use a descriptive `User-Agent`, and call `time.sleep(1)` after your request.

**Hint:** `BeautifulSoup`, `soup.select("article.product_pod")`, then pull the title from `h3 a` (the `title` attribute) and the price from `p.price_color` within each matched element.


In [47]:
from bs4 import BeautifulSoup

def scrape_book_titles_and_prices():
    """
    returns: pandas DataFrame with columns title, price
    """
    url = "http://books.toscrape.com"
    headers = {"User-Agent": "KDU-DataScience-Student"}
    # TODO: implement — request, parse with BeautifulSoup, extract title + price, time.sleep(1)
    pass


from bs4 import BeautifulSoup
import requests
import pandas as pd
import time

def scrape_book_titles_and_prices():
    url = "https://books.toscrape.com/"
    headers = {"User-Agent": "KDU-DataScience-Student"}

    response = requests.get(
        url, headers=headers, timeout=15
    )
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    rows = []

    for book in soup.select("article.product_pod"):
        title_element = book.select_one("h3 a")
        price_element = book.select_one("p.price_color")

        if title_element and price_element:
            rows.append({
                "title": title_element.get("title", ""),
                "price": price_element.get_text(strip=True)
            })

    time.sleep(1)
    return pd.DataFrame(rows)

---
## Cleanup
Run this last to close your database connection.

In [48]:
conn.close()
print("Connection closed.")

Connection closed.
